# Linear Regression Lab

In [15]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import *
import pandas as pd

In [2]:
spark=SparkSession.builder.appName('House_Price_Prediction').master('local[*]').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 11:35:16 WARN Utils: Your hostname, Samriddhas-Macbook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.1.8.149 instead (on interface en0)
26/06/14 11:35:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 11:35:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 11:35:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Load data from SkLearn

In [3]:
from sklearn.datasets import fetch_california_housing

In [4]:
housing=fetch_california_housing()
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]]),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': '.. _california_housing_dataset:\n

In [5]:
df_pandas=pd.DataFrame(housing.data,columns=housing.feature_names)
df_pandas.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [6]:
df_pandas['MedHouseVal']=housing.target
df_pandas

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


In [7]:
df=spark.createDataFrame(df_pandas)
df.show(5)

[Stage 0:>                                                          (0 + 1) / 1]

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|MedHouseVal|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23|      4.526|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22|      3.585|
|7.2574|    52.0| 8.288135593220339| 1.073446327683616|     496.0|2.8022598870056497|   37.85|  -122.24|      3.521|
|5.6431|    52.0|5.8173515981735155|1.0730593607305936|     558.0| 2.547945205479452|   37.85|  -122.25|      3.413|
|3.8462|    52.0| 6.281853281853282|1.0810810810810811|     565.0|2.1814671814671813|   37.85|  -122.25|      3.422|
+------+--------+------------------+------------------+---------

Traceback (most recent call last):
  File "/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe
                                                                                

In [8]:
print(df.count())
print(len(df.columns))

20640
9


In [12]:
df.distinct().count()

df_clean=df.distinct()

df_clean.describe('AveRooms').show()

+-------+------------------+
|summary|          AveRooms|
+-------+------------------+
|  count|             20640|
|   mean|  5.42899974219036|
| stddev|2.4741731394243165|
|    min|0.8461538461538461|
|    max| 141.9090909090909|
+-------+------------------+



In [13]:
df_features=df_clean.withColumn(
    "RoomsPerHousehold",
    col('AveRooms') / col('AveBedrms')
).withColumn(
    "PopulationPerHousehold",
    col('Population')/col("AveOccup")
)

df_features.printSchema()

root
 |-- MedInc: double (nullable = true)
 |-- HouseAge: double (nullable = true)
 |-- AveRooms: double (nullable = true)
 |-- AveBedrms: double (nullable = true)
 |-- Population: double (nullable = true)
 |-- AveOccup: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- MedHouseVal: double (nullable = true)
 |-- RoomsPerHousehold: double (nullable = true)
 |-- PopulationPerHousehold: double (nullable = true)



In [14]:
feature_cols=[
    "MedInc","HouseAge","AveRooms","AveBedrms","Population","AveOccup","Latitude","Longitude","RoomsPerHousehold","PopulationPerHousehold"
]

target_col="MedHouseVal"

In [17]:
assembler=VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_assembler=assembler.transform(df_features)
df_assembler.show()

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+------------------+----------------------+--------------------+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|MedHouseVal| RoomsPerHousehold|PopulationPerHousehold|            features|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+------------------+----------------------+--------------------+
|  3.12|    52.0| 4.797527047913447| 1.061823802163833|    1157.0|1.7882534775888717|   37.84|  -122.25|      2.414| 4.518195050946143|                 647.0|[3.12,52.0,4.7975...|
|7.0175|    37.0| 6.982954545454546|1.0284090909090908|     420.0|2.3863636363636362|   37.82|   -122.2|      3.667| 6.790055248618786|                 176.0|[7.0175,37.0,6.98...|
|5.8948|    52.0| 7.740088105726873|1.0837004405286343|     585.0|2.5770925110132157|   37.83|  -122

In [22]:
scaler=StandardScaler(
    inputCol="features",
    outputCol="sacledFeatures",
    withMean=True,
    withStd=True
)

scaler_model=scaler.fit(df_assembler)
df_scaled=scaler_model.transform(df_assembler)

In [23]:
train_df,test_df=df_scaled.randomSplit([0.8,0.2],seed=42)

In [24]:
train_df.count()

16560